In [0]:
# imports
import pyspark.sql.functions as F

In [0]:
# gold class
class GoldAggregations:
    def __init__(self, catalog_name, silver_table_schema, silver_table_name, gold_schema, gold_aggregated_table_name, gold_feature_table_name):
        self.catalog_name = catalog_name
        self.silver_table_schema = silver_table_schema
        self.silver_table_name = silver_table_name
        self.gold_schema = gold_schema
        self.gold_aggregated_table_name = gold_aggregated_table_name
        self.gold_feature_table_name = gold_feature_table_name

    def build_gold_aggregated_table(self):
        silver_table_name = self.catalog_name + '.' + self.silver_table_schema + '.' +  self.silver_table_name
        gold_table_name = self.catalog_name + '.' + self.gold_schema + '.' +  self.gold_aggregated_table_name

        print(f"03: Building Gold Aggregated Table, Path: {gold_table_name}")
        spark.sql(f"""
            CREATE OR REPLACE TABLE {gold_table_name} AS
            SELECT 
                transaction_date,
                product_name,
                destination_city,
                ROUND(avg_supplier_reliability_score, 2) AS avg_supplier_reliability_score,
                total_ordered_quantity,
                ROUND(total_demand_quantity, 2) AS total_demand_quantity,
                total_available_inventory,
                ROUND(total_unit_price_usd, 2) AS total_unit_price_usd,
                ROUND(total_product_cost_usd, 2) AS total_product_cost_usd,
                ROUND(total_transportation_cost_usd, 2) AS total_transportation_cost_usd,
                ROUND(total_cost_usd, 2) AS total_cost_usd,
                total_expected_lead_time_days,
                total_actual_lead_time_days,
                delay_days,
                is_delayed,
                is_stockout,
                ROUND(avg_quality_score, 2) AS avg_quality_score,
                CURRENT_TIMESTAMP() AS gold_ingestion_timestamp
            FROM (
                SELECT 
                    transaction_date,
                    product_name,
                    destination_city,
                    AVG(supplier_reliability_score) AS avg_supplier_reliability_score,
                    SUM(ordered_quantity) AS total_ordered_quantity,
                    SUM(demand_quantity) AS total_demand_quantity,
                    SUM(available_inventory) AS total_available_inventory,
                    SUM(unit_price_usd) AS total_unit_price_usd,
                    SUM(product_cost_usd) AS total_product_cost_usd,
                    SUM(transportation_cost_usd) AS total_transportation_cost_usd,
                    SUM(total_cost_usd) AS total_cost_usd,
                    SUM(expected_lead_time_days) AS total_expected_lead_time_days,
                    SUM(actual_lead_time_days) AS total_actual_lead_time_days,
                    total_expected_lead_time_days - total_actual_lead_time_days AS delay_days,
                    CASE
                        WHEN (total_expected_lead_time_days - total_actual_lead_time_days) < 0 THEN 0
                        ELSE 1
                    END AS is_delayed,
                    CASE
                        WHEN total_ordered_quantity = total_available_inventory THEN 0
                        ELSE 1
                    END AS is_stockout,
                    AVG(quality_score) AS avg_quality_score
                FROM {silver_table_name}
                GROUP BY transaction_date,
                        product_name,
                        destination_city
                ) AS aggregated_data
                """)
        print(f"04: Gold Aggregated Table Created")

    def build_gold_feature_table(self):
        pass

    def read_silver_table(self):
        silver_table_path = self.catalog_name + '.' + self.silver_table_schema + '.' + self.silver_table_name
        print(f"01: Loading the silver table, path {silver_table_path}")
        self.silver_data = spark.read.table(silver_table_path)
        print("02: Silver table loaded")



In [0]:
# main part
gold_aggregations = GoldAggregations(
    catalog_name="supply_chain",
    silver_table_schema="silver",
    silver_table_name="silver_supply_chain",
    gold_schema="gold",
    gold_aggregated_table_name="aggregated_supply_chain",
    gold_feature_table_name="feature_supply_chain"
)

gold_aggregations.read_silver_table()
gold_aggregations.build_gold_aggregated_table()

In [0]:
%sql
SELECT * 
FROM supply_chain.gold.aggregated_supply_chain;

In [0]:
%sql
SELECT COUNT(*) AS aggregated_table_count
FROM supply_chain.gold.aggregated_supply_chain;